# QStorePrice AI — Training Notebook

**What this does**: Trains a Qwen-2.5-7B model to manage a perishable goods store.

**Time needed**: ~3-4 hours total on a T4 GPU.

**Before you start**: Make sure you selected **GPU runtime**:
- Runtime → Change runtime type → T4 GPU

## Cell 1: Install Dependencies (~5 min)

In [ ]:
# Install Unsloth first (must come before other packages)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Install training dependencies
!pip install torch transformers trl peft accelerate bitsandbytes -q
!pip install datasets tokenizers gymnasium wandb numpy pandas scipy tqdm -q
!pip install matplotlib gradio -q

print("\n=== Dependencies installed ===")

## Cell 2: Clone Repo & Verify GPU

In [ ]:
!git clone https://github.com/nandeshkanagaraju/QStorePrice.git
%cd QStorePrice

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

## Cell 3: Run All 4 Verification Checks

In [ ]:
# Check 1: Imports
from freshprice_env.enums import *
from freshprice_env.constants import *
from freshprice_env.entities import *
from freshprice_env.market_state import MarketStateBuilder
from freshprice_env.engines.pricing_engine import PricingEngine
from freshprice_env.engines.farmer_engine import FarmerEngine
from freshprice_env.engines.trend_engine import TrendEngine
from freshprice_env.reward import WRRRewardEngine
from freshprice_env.brief_pipeline.prompt_builder import OperatingBriefPromptBuilder
from freshprice_env.brief_pipeline.parser import BriefParser
from freshprice_env.brief_pipeline.validator import BriefValidator
from freshprice_env.brief_pipeline.quality_scorer import BriefQualityScorer
from freshprice_env.brief_pipeline.rule_executor import RuleExecutor
from freshprice_env.freshprice_env import FreshPriceEnv
from training.curriculum import CurriculumManager, EpisodeResult
from training.trajectory_buffer import TrajectoryBuffer
from training.counterfactual import CounterfactualEngine
print('CHECK 1: ALL IMPORTS OK')

# Check 2: Gym
from gymnasium.utils.env_checker import check_env
env = FreshPriceEnv()
check_env(env)
print('CHECK 2: GYM CHECK PASSED')

# Check 3: Smoke test
from freshprice_env.enums import CurriculumScenario
class DummyLLM:
    def generate(self, prompt):
        return 'SITUATION: Testing.\nSIGNAL ANALYSIS: N/A\nVIABILITY CHECK: N/A\nRECOMMENDATION: Hold.\nDIRECTIVE: {"engine": "PRICING", "actions": []}\nCONFIDENCE: LOW'

env = FreshPriceEnv(scenario=CurriculumScenario.STABLE_WEEK, seed=42, llm_client=DummyLLM())
obs, info = env.reset()
done = False
steps = 0
while not done:
    obs, reward, done, truncated, info = env.step(DummyLLM().generate(obs))
    steps += 1
print(f'CHECK 3: SMOKE TEST PASSED — {steps} steps, WRR: {info.get("final_reward", {}).get("wrr", 0.0):.4f}')

# Check 4: WRR weights
from freshprice_env.constants import WRR_WEIGHT_R1, WRR_WEIGHT_R2, WRR_WEIGHT_R3
total = WRR_WEIGHT_R1 + WRR_WEIGHT_R2 + WRR_WEIGHT_R3
assert abs(total - 1.0) < 1e-9
print(f'CHECK 4: WRR WEIGHTS OK: {WRR_WEIGHT_R1} + {WRR_WEIGHT_R2} + {WRR_WEIGHT_R3} = {total}')

print('\n=== ALL 4 CHECKS PASSED ===')

## Cell 4: Login to WandB

Get your API key from https://wandb.ai/authorize

In [ ]:
import wandb
wandb.login()

## Cell 5: Test the Base Model (Zero-Shot — BEFORE Training)

This runs the unmodified Qwen-2.5-7B on 3 episodes to see how it performs **without any training**.

Expected: WRR around 0.0-0.10 (terrible — the model doesn't know what an Operating Brief is yet).

**Time: ~15-20 min**

In [ ]:
import torch
from unsloth import FastLanguageModel
from freshprice_env.freshprice_env import FreshPriceEnv
from freshprice_env.enums import CurriculumScenario

# Load base model
print("Loading Qwen-2.5-7B-Instruct (zero-shot, no training)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("Model loaded.")

def generate_brief(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=800, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# Run 3 episodes on STABLE_WEEK
baseline_results = []
for i in range(3):
    env = FreshPriceEnv(scenario=CurriculumScenario.STABLE_WEEK, seed=1000+i)
    obs, info = env.reset(seed=1000+i)
    done = False
    step = 0
    while not done:
        brief = generate_brief(obs)
        obs, reward, done, truncated, info = env.step(brief)
        step += 1
        if step % 20 == 0:
            print(f"  Episode {i+1}, step {step}/84...")

    final = info.get("final_reward", {})
    wrr = final.get("wrr", 0.0)
    quality = final.get("brief_quality_score", 0.0)
    violations = final.get("anti_hack_violations", 0)
    baseline_results.append({"wrr": wrr, "quality": quality, "violations": violations})
    print(f"Episode {i+1}: WRR={wrr:.4f} | Quality={quality:.3f} | Violations={violations}")

avg_wrr = sum(r["wrr"] for r in baseline_results) / len(baseline_results)
avg_quality = sum(r["quality"] for r in baseline_results) / len(baseline_results)
print(f"\n=== BASELINE (Zero-Shot) ===")
print(f"  Average WRR:     {avg_wrr:.4f}")
print(f"  Average Quality: {avg_quality:.3f}")
print(f"  (This is the 'before' — training should improve these numbers)")

## Cell 6: Generate SFT Training Data

Before SFT, we need hand-crafted examples. This generates 60 examples across all 3 engine types.

**Time: instant**

In [ ]:
import json
import os

os.makedirs("training/sft_data", exist_ok=True)

# ── PRICING BRIEFS (25 examples) ──
pricing_briefs = []

# CRITICAL urgency — must discount aggressively
for i in range(5):
    pricing_briefs.append({
        "prompt": f"=== CURRENT INVENTORY ===\n[🔴] dairy — Batch batch_{i:04d}\n  Stock: {10+i*3} units | Expiry: {2.0+i*0.5:.1f}hrs | Urgency: CRITICAL\n  Current price: Rs 75 | Original: Rs 75 | Floor: Rs 32\n  Velocity: 1.20 units/hr | Discount: 0%\n\n=== MARKET CONTEXT ===\nDay: Wednesday | Hour: 2:00 PM | Risk Buffer: Rs 5000\n\n=== YOUR TASK ===\nWrite a PRICING Operating Brief.",
        "completion": f"SITUATION: Dairy batch batch_{i:04d} is CRITICAL with only {2.0+i*0.5:.1f} hours remaining and {10+i*3} units unsold. Current velocity of 1.20 units/hr is insufficient to clear stock at full price. Aggressive discounting required immediately.\nSIGNAL ANALYSIS: N/A\nVIABILITY CHECK: N/A\nRECOMMENDATION: Apply deep discount to 0.40x multiplier and trigger flash sale to maximise clearance before expiry.\nDIRECTIVE: {{\"engine\": \"PRICING\", \"actions\": [{{\"batch_id\": \"batch_{i:04d}\", \"price_multiplier\": 0.40, \"flash_sale\": true, \"bundle_with\": null}}]}}\nCONFIDENCE: HIGH",
        "engine_type": "PRICING", "scenario": "STABLE_WEEK", "difficulty": "easy",
        "notes": "CRITICAL batch needs aggressive discount + flash sale"
    })

# URGENT urgency — moderate discount
for i in range(5):
    pricing_briefs.append({
        "prompt": f"=== CURRENT INVENTORY ===\n[🟠] vegetables — Batch batch_{10+i:04d}\n  Stock: {15+i*2} units | Expiry: {8.0+i*2:.1f}hrs | Urgency: URGENT\n  Current price: Rs 45 | Original: Rs 45 | Floor: Rs 15\n  Velocity: 2.00 units/hr | Discount: 0%\n\n=== MARKET CONTEXT ===\nDay: Friday | Hour: 5:00 PM | Risk Buffer: Rs 4500\n\n=== YOUR TASK ===\nWrite a PRICING Operating Brief.",
        "completion": f"SITUATION: Vegetable batch batch_{10+i:04d} is URGENT with {8.0+i*2:.1f} hours remaining. Current velocity of 2.00 units/hr needs a boost to clear {15+i*2} units before expiry. Evening peak demand provides an opportunity.\nSIGNAL ANALYSIS: N/A\nVIABILITY CHECK: N/A\nRECOMMENDATION: Moderate discount to 0.70x to increase demand velocity during evening peak hours.\nDIRECTIVE: {{\"engine\": \"PRICING\", \"actions\": [{{\"batch_id\": \"batch_{10+i:04d}\", \"price_multiplier\": 0.70, \"flash_sale\": false, \"bundle_with\": null}}]}}\nCONFIDENCE: HIGH",
        "engine_type": "PRICING", "scenario": "STABLE_WEEK", "difficulty": "easy",
        "notes": "URGENT batch needs moderate discount"
    })

# WATCH urgency — hold or slight discount
for i in range(5):
    pricing_briefs.append({
        "prompt": f"=== CURRENT INVENTORY ===\n[🟡] fruits — Batch batch_{20+i:04d}\n  Stock: {20+i} units | Expiry: {36.0+i*5:.1f}hrs | Urgency: WATCH\n  Current price: Rs 60 | Original: Rs 60 | Floor: Rs 25\n  Velocity: 2.50 units/hr | Discount: 0%\n\n=== MARKET CONTEXT ===\nDay: Tuesday | Hour: 10:00 AM | Risk Buffer: Rs 5000\n\n=== YOUR TASK ===\nWrite a PRICING Operating Brief.",
        "completion": f"SITUATION: Fruit batch batch_{20+i:04d} is at WATCH urgency with {36.0+i*5:.1f} hours remaining and {20+i} units. Current velocity of 2.50 units/hr is adequate for sell-through. No immediate discount needed.\nSIGNAL ANALYSIS: N/A\nVIABILITY CHECK: N/A\nRECOMMENDATION: Hold at current price. Velocity is sufficient to clear inventory before urgency escalates. Monitor and reassess in next brief cycle.\nDIRECTIVE: {{\"engine\": \"PRICING\", \"actions\": [{{\"batch_id\": \"batch_{20+i:04d}\", \"price_multiplier\": 1.0, \"flash_sale\": false, \"bundle_with\": null}}]}}\nCONFIDENCE: HIGH",
        "engine_type": "PRICING", "scenario": "STABLE_WEEK", "difficulty": "medium",
        "notes": "WATCH batch — correct answer is to hold price, not discount"
    })

# FRESH — definitely hold
for i in range(5):
    pricing_briefs.append({
        "prompt": f"=== CURRENT INVENTORY ===\n[🟢] packaged — Batch batch_{30+i:04d}\n  Stock: {30+i*5} units | Expiry: {96.0+i*10:.1f}hrs | Urgency: FRESH\n  Current price: Rs 100 | Original: Rs 100 | Floor: Rs 50\n  Velocity: 1.50 units/hr | Discount: 0%\n\n=== MARKET CONTEXT ===\nDay: Monday | Hour: 9:00 AM | Risk Buffer: Rs 5000\n\n=== YOUR TASK ===\nWrite a PRICING Operating Brief.",
        "completion": f"SITUATION: Packaged goods batch batch_{30+i:04d} is FRESH with {96.0+i*10:.1f} hours remaining. No urgency. {30+i*5} units at standard velocity of 1.50 units/hr will clear well within shelf life.\nSIGNAL ANALYSIS: N/A\nVIABILITY CHECK: N/A\nRECOMMENDATION: Hold at full price. Over 4 days of shelf life remaining. Early discounting wastes margin.\nDIRECTIVE: {{\"engine\": \"PRICING\", \"actions\": [{{\"batch_id\": \"batch_{30+i:04d}\", \"price_multiplier\": 1.0, \"flash_sale\": false, \"bundle_with\": null}}]}}\nCONFIDENCE: HIGH",
        "engine_type": "PRICING", "scenario": "STABLE_WEEK", "difficulty": "easy",
        "notes": "FRESH batch — hold price, no discount needed"
    })

# Override examples — naive choice is wrong
for i in range(5):
    pricing_briefs.append({
        "prompt": f"=== CURRENT INVENTORY ===\n[🟡] herbs — Batch batch_{40+i:04d}\n  Stock: 3 units | Expiry: {28.0+i*2:.1f}hrs | Urgency: WATCH\n  Current price: Rs 25 | Original: Rs 25 | Floor: Rs 10\n  Velocity: 0.80 units/hr | Discount: 0%\n\n=== MARKET CONTEXT ===\nDay: Saturday | Hour: 11:00 AM | Risk Buffer: Rs 5000\n\n=== YOUR TASK ===\nWrite a PRICING Operating Brief.",
        "completion": f"SITUATION: Herb batch batch_{40+i:04d} at WATCH urgency but only 3 units remaining with {28.0+i*2:.1f} hours. At current velocity of 0.80 units/hr, these 3 units will sell in ~4 hours. Saturday morning demand is strong.\nSIGNAL ANALYSIS: N/A\nVIABILITY CHECK: N/A\nRECOMMENDATION: Hold at full price despite WATCH urgency. Low quantity + weekend demand means no discount needed. Discounting 3 units wastes margin for negligible waste risk.\nDIRECTIVE: {{\"engine\": \"PRICING\", \"actions\": [{{\"batch_id\": \"batch_{40+i:04d}\", \"price_multiplier\": 1.0, \"flash_sale\": false, \"bundle_with\": null}}]}}\nCONFIDENCE: HIGH",
        "engine_type": "PRICING", "scenario": "STABLE_WEEK", "difficulty": "hard",
        "notes": "Override: WATCH urgency but low stock + weekend = hold price"
    })

with open("training/sft_data/pricing_briefs.json", "w") as f:
    json.dump(pricing_briefs, f, indent=2)
print(f"Pricing briefs: {len(pricing_briefs)} examples")

# ── FARMER BRIEFS (20 examples) ──
farmer_briefs = []

# High viability — ACCEPT
for i in range(5):
    farmer_briefs.append({
        "prompt": f"=== PENDING FARMER OFFERS ===\nOffer offer_{i:03d}: Farmer Rajan — {40+i*5}kg of mangoes\n  Offered price: Rs {30+i*2}/kg\n  Seller-reported shelf life: 48hrs\n  Viability score: {0.80+i*0.02:.2f} (Strong Accept)\n\n=== RISK BUFFER ===\nCurrent balance: Rs 4500\n\n=== YOUR TASK ===\nWrite a FARMER Operating Brief.",
        "completion": f"SITUATION: Farmer Rajan offers {40+i*5}kg of mangoes at Rs {30+i*2}/kg with 48hrs shelf life. Viability score {0.80+i*0.02:.2f} is strong. Risk buffer healthy at Rs 4,500.\nSIGNAL ANALYSIS: N/A\nVIABILITY CHECK: Shelf Life: PASS — 48hrs covers projected sell-through. Inventory Conflict: PASS — no urgent mango stock. Break-Even: PASS — market price Rs 75/kg vs cost Rs {30+i*2+8}/kg. Worst-Case P&L: PASS — 60% sell-through covers cost. Demand Timing: PASS — good demand.\nRECOMMENDATION: ACCEPT at offered price. Viability {0.80+i*0.02:.2f} with healthy buffer supports acceptance.\nDIRECTIVE: {{\"engine\": \"FARMER\", \"actions\": [{{\"offer_id\": \"offer_{i:03d}\", \"decision\": \"ACCEPT\", \"counter_price\": null}}]}}\nCONFIDENCE: HIGH",
        "engine_type": "FARMER", "scenario": "FARMER_WEEK", "difficulty": "easy",
        "notes": "High viability offer — straightforward accept"
    })

# Medium viability — COUNTER
for i in range(5):
    farmer_briefs.append({
        "prompt": f"=== PENDING FARMER OFFERS ===\nOffer offer_{10+i:03d}: Farmer Lakshmi — {20+i*3}kg of spinach\n  Offered price: Rs {18+i}/kg\n  Seller-reported shelf life: 30hrs\n  Viability score: {0.58+i*0.02:.2f} (Acceptable)\n\n=== RISK BUFFER ===\nCurrent balance: Rs 3800\n\n=== YOUR TASK ===\nWrite a FARMER Operating Brief.",
        "completion": f"SITUATION: Farmer Lakshmi offers {20+i*3}kg of spinach at Rs {18+i}/kg with 30hrs shelf life. Viability {0.58+i*0.02:.2f} is acceptable but not strong. Buffer at Rs 3,800 is adequate.\nSIGNAL ANALYSIS: N/A\nVIABILITY CHECK: Shelf Life: FLAG — 30hrs is tight for {20+i*3}kg at current velocity. Inventory Conflict: PASS — no urgent spinach stock. Break-Even: PASS — margin adequate. Worst-Case P&L: FLAG — tight margins at 60% sell-through. Demand Timing: PASS.\nRECOMMENDATION: COUNTER at Rs {int((18+i)*0.90)}/kg to improve margins given the tight shelf life.\nDIRECTIVE: {{\"engine\": \"FARMER\", \"actions\": [{{\"offer_id\": \"offer_{10+i:03d}\", \"decision\": \"COUNTER\", \"counter_price\": {(18+i)*0.90:.1f}}}]}}\nCONFIDENCE: MEDIUM",
        "engine_type": "FARMER", "scenario": "FARMER_WEEK", "difficulty": "medium",
        "notes": "Medium viability — counter for better price"
    })

# Low viability — DECLINE
for i in range(5):
    farmer_briefs.append({
        "prompt": f"=== PENDING FARMER OFFERS ===\nOffer offer_{20+i:03d}: Farmer Suresh — {30+i*5}kg of tomatoes\n  Offered price: Rs {25+i*3}/kg\n  Seller-reported shelf life: 18hrs\n  Viability score: {0.25+i*0.03:.2f} (High Risk)\n\n=== RISK BUFFER ===\nCurrent balance: Rs 2200\n\n=== YOUR TASK ===\nWrite a FARMER Operating Brief.",
        "completion": f"SITUATION: Farmer Suresh offers {30+i*5}kg of tomatoes at Rs {25+i*3}/kg with only 18hrs shelf life. Viability {0.25+i*0.03:.2f} is high risk. Buffer at Rs 2,200 is low.\nSIGNAL ANALYSIS: N/A\nVIABILITY CHECK: Shelf Life: FAIL — 18hrs insufficient for {30+i*5}kg. Inventory Conflict: PASS. Break-Even: FLAG — tight at this price. Worst-Case P&L: FAIL — losses likely. Demand Timing: PASS.\nRECOMMENDATION: DECLINE. Viability {0.25+i*0.03:.2f} is below acceptable threshold. Short shelf life and low buffer make this too risky.\nDIRECTIVE: {{\"engine\": \"FARMER\", \"actions\": [{{\"offer_id\": \"offer_{20+i:03d}\", \"decision\": \"DECLINE\", \"counter_price\": null}}]}}\nCONFIDENCE: HIGH",
        "engine_type": "FARMER", "scenario": "FARMER_WEEK", "difficulty": "easy",
        "notes": "Low viability + low buffer — clear decline"
    })

# Override: high viability but conflicting urgent stock — DECLINE
for i in range(5):
    farmer_briefs.append({
        "prompt": f"=== PENDING FARMER OFFERS ===\nOffer offer_{30+i:03d}: Farmer Priya — 25kg of dairy\n  Offered price: Rs 40/kg\n  Seller-reported shelf life: 36hrs\n  Viability score: 0.75 (Acceptable)\n\n=== CURRENT INVENTORY (same category) ===\n[🔴] dairy — Batch batch_existing | 18 units | 4.5hrs | Rs 65\n\n=== RISK BUFFER ===\nCurrent balance: Rs 1800\n\n=== YOUR TASK ===\nWrite a FARMER Operating Brief.",
        "completion": f"SITUATION: Farmer Priya offers 25kg dairy at Rs 40/kg with 36hrs shelf life. Viability 0.75 looks acceptable BUT we have 18 units of CRITICAL dairy expiring in 4.5 hours. Buffer low at Rs 1,800.\nSIGNAL ANALYSIS: N/A\nVIABILITY CHECK: Shelf Life: PASS — 36hrs adequate. Inventory Conflict: FAIL — CRITICAL dairy batch already active. Break-Even: PASS. Worst-Case P&L: FLAG — low buffer. Demand Timing: PASS.\nRECOMMENDATION: DECLINE despite 0.75 viability. Existing CRITICAL dairy stock must be cleared first. Accepting more dairy would compound the problem.\nDIRECTIVE: {{\"engine\": \"FARMER\", \"actions\": [{{\"offer_id\": \"offer_{30+i:03d}\", \"decision\": \"DECLINE\", \"counter_price\": null}}]}}\nCONFIDENCE: HIGH",
        "engine_type": "FARMER", "scenario": "FARMER_WEEK", "difficulty": "hard",
        "notes": "Override: viability looks good but inventory conflict makes it wrong to accept"
    })

with open("training/sft_data/farmer_briefs.json", "w") as f:
    json.dump(farmer_briefs, f, indent=2)
print(f"Farmer briefs: {len(farmer_briefs)} examples")

# ── TREND BRIEFS (15 examples) ──
trend_briefs = []

# High confidence — APPROVE
for i in range(5):
    trend_briefs.append({
        "prompt": f"=== ACTIVE TREND SIGNALS ===\n📱 MUSHROOMS — Score: {80+i*2}/100 (INSTAGRAM)\n  Suggested restock: {8+i*2}kg\n  Signal factors:\n    Recipe simplicity:     0.85\n    Ingredient rarity:     0.80\n    View velocity:         0.82\n    Local relevance:       0.70\n    Historical conversion: 0.65\n  Confidence tier: HIGH\n\n=== YOUR TASK ===\nWrite a TREND Operating Brief.",
        "completion": f"SITUATION: Strong Instagram trend signal for mushrooms with composite score {80+i*2}/100. High recipe simplicity and view velocity suggest this is a viral recipe moment. Suggested restock {8+i*2}kg.\nSIGNAL ANALYSIS: Composite score {80+i*2} is well above 65 threshold. Recipe simplicity 0.85 means customers can actually make the dish. Ingredient rarity 0.80 means mushrooms are a key ingredient. View velocity 0.82 shows rapid growth. Historical conversion 0.65 suggests moderate but positive past performance.\nVIABILITY CHECK: Recipe Simplicity: PASS — 0.85. Ingredient Rarity: PASS — 0.80. View Velocity: PASS — 0.82. Local Relevance: PASS — 0.70. Historical Conversion: PASS — 0.65.\nRECOMMENDATION: APPROVE full suggested quantity of {8+i*2}kg. Score {80+i*2} with strong factors across all dimensions.\nDIRECTIVE: {{\"engine\": \"TREND\", \"actions\": [{{\"category\": \"mushrooms\", \"decision\": \"APPROVE\", \"order_quantity_kg\": {8+i*2}.0}}]}}\nCONFIDENCE: HIGH",
        "engine_type": "TREND", "scenario": "TREND_WEEK", "difficulty": "easy",
        "notes": "High confidence trend — straightforward approve"
    })

# Medium confidence — APPROVE conservative
for i in range(5):
    trend_briefs.append({
        "prompt": f"=== ACTIVE TREND SIGNALS ===\n📱 HERBS — Score: {67+i}/100 (YOUTUBE)\n  Suggested restock: {10+i}kg\n  Signal factors:\n    Recipe simplicity:     0.70\n    Ingredient rarity:     0.65\n    View velocity:         0.68\n    Local relevance:       0.60\n    Historical conversion: 0.62\n  Confidence tier: MEDIUM\n\n=== YOUR TASK ===\nWrite a TREND Operating Brief.",
        "completion": f"SITUATION: YouTube trend for herbs with composite score {67+i}/100. Above threshold but not high confidence. Moderate factors across the board.\nSIGNAL ANALYSIS: Composite {67+i} is above 65 threshold but below 80 high-confidence mark. Historical conversion 0.62 provides some evidence of past demand conversion. View velocity 0.68 is moderate — trend may not sustain.\nVIABILITY CHECK: Recipe Simplicity: PASS — 0.70. Ingredient Rarity: FLAG — 0.65. View Velocity: FLAG — 0.68. Local Relevance: FLAG — 0.60. Historical Conversion: PASS — 0.62.\nRECOMMENDATION: APPROVE at conservative quantity ({int((10+i)*0.75)}kg, 75% of suggested) given moderate signal strength.\nDIRECTIVE: {{\"engine\": \"TREND\", \"actions\": [{{\"category\": \"herbs\", \"decision\": \"APPROVE\", \"order_quantity_kg\": {(10+i)*0.75:.1f}}}]}}\nCONFIDENCE: MEDIUM",
        "engine_type": "TREND", "scenario": "TREND_WEEK", "difficulty": "medium",
        "notes": "Medium confidence — approve but conservative quantity"
    })

# Below threshold or weak — DECLINE
for i in range(5):
    trend_briefs.append({
        "prompt": f"=== ACTIVE TREND SIGNALS ===\n📱 BAKERY — Score: {60+i}/100 (ZOMATO)\n  Suggested restock: 12kg\n  Signal factors:\n    Recipe simplicity:     0.55\n    Ingredient rarity:     0.50\n    View velocity:         0.60\n    Local relevance:       0.55\n    Historical conversion: 0.40\n  Confidence tier: LOW\n\n=== YOUR TASK ===\nWrite a TREND Operating Brief.",
        "completion": f"SITUATION: Zomato signal for bakery items with composite score {60+i}/100. Below or barely at threshold. Historical conversion 0.40 is weak — past trends in this category did not convert to sales.\nSIGNAL ANALYSIS: Composite {60+i} {'is below' if 60+i < 65 else 'barely meets'} the 65 threshold. Low historical conversion (0.40) is the key concern — bakery trends on Zomato have not reliably driven purchases in the past. Ingredient rarity 0.50 suggests these items are easily available elsewhere.\nVIABILITY CHECK: Recipe Simplicity: FLAG — 0.55. Ingredient Rarity: FAIL — 0.50. View Velocity: FLAG — 0.60. Local Relevance: FLAG — 0.55. Historical Conversion: FAIL — 0.40.\nRECOMMENDATION: DECLINE. Weak signal with poor historical conversion. Risk of overstocking outweighs potential demand uplift.\nDIRECTIVE: {{\"engine\": \"TREND\", \"actions\": [{{\"category\": \"bakery\", \"decision\": \"DECLINE\", \"order_quantity_kg\": null}}]}}\nCONFIDENCE: HIGH",
        "engine_type": "TREND", "scenario": "TREND_WEEK", "difficulty": "easy",
        "notes": "Weak signal — decline"
    })

with open("training/sft_data/trend_briefs.json", "w") as f:
    json.dump(trend_briefs, f, indent=2)
print(f"Trend briefs: {len(trend_briefs)} examples")

print(f"\n=== Total SFT examples: {len(pricing_briefs) + len(farmer_briefs) + len(trend_briefs)} ===")

## Cell 7: Run SFT Warm-Start (~30-45 min)

This teaches the model the Operating Brief format so GRPO starts with non-garbage output.

**What to watch**: Loss should drop from ~2.5 to ~0.8

In [ ]:
!python training/sft_trainer.py \
    --model-id Qwen/Qwen2.5-7B-Instruct \
    --output-dir checkpoints/sft_v1 \
    --data-dir training/sft_data \
    --epochs 2 \
    --seed 42

## Cell 8: Test After SFT (~15 min)

Run the same 3 episodes as baseline but with the SFT-trained model.

**Expected**: WRR should be higher than zero-shot (maybe 0.05-0.20). Brief quality should be notably better.

In [ ]:
import torch
from unsloth import FastLanguageModel
from freshprice_env.freshprice_env import FreshPriceEnv
from freshprice_env.enums import CurriculumScenario

# Load SFT checkpoint
print("Loading SFT checkpoint...")
model_sft, tokenizer_sft = FastLanguageModel.from_pretrained(
    model_name="checkpoints/sft_v1",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model_sft)

def generate_sft(prompt):
    inputs = tokenizer_sft(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model_sft.device)
    with torch.no_grad():
        outputs = model_sft.generate(**inputs, max_new_tokens=800, do_sample=False, pad_token_id=tokenizer_sft.eos_token_id)
    return tokenizer_sft.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

sft_results = []
for i in range(3):
    env = FreshPriceEnv(scenario=CurriculumScenario.STABLE_WEEK, seed=1000+i)
    obs, info = env.reset(seed=1000+i)
    done = False
    step = 0
    while not done:
        brief = generate_sft(obs)
        obs, reward, done, truncated, info = env.step(brief)
        step += 1
        if step % 20 == 0:
            print(f"  Episode {i+1}, step {step}/84...")
    final = info.get("final_reward", {})
    wrr = final.get("wrr", 0.0)
    quality = final.get("brief_quality_score", 0.0)
    sft_results.append({"wrr": wrr, "quality": quality})
    print(f"Episode {i+1}: WRR={wrr:.4f} | Quality={quality:.3f}")

print(f"\n=== COMPARISON ===")
print(f"{'':>20} {'WRR':>8} {'Quality':>10}")
print(f"{'Zero-shot':>20} {sum(r['wrr'] for r in baseline_results)/3:>8.4f} {sum(r['quality'] for r in baseline_results)/3:>10.3f}")
print(f"{'After SFT':>20} {sum(r['wrr'] for r in sft_results)/3:>8.4f} {sum(r['quality'] for r in sft_results)/3:>10.3f}")

## Cell 9: GRPO Training (~2-3 hours)

This is the main RL training. The model learns from episode outcomes.

**Watch WandB** for wrr and brief_quality_score rising together.

**Note**: This takes a long time. If Colab disconnects, use `--resume-from` to continue.

In [ ]:
!python training/train.py \
    --resume-from checkpoints/sft_v1 \
    --output-dir checkpoints \
    --start-scenario STABLE_WEEK \
    --episodes-per-level 50 \
    --dpo-every-n-episodes 25 \
    --wandb-project qstoreprice-ai \
    --seed 42

## Cell 10: Final Evaluation

Compare the trained model against the baseline.

In [ ]:
import torch
import glob
from unsloth import FastLanguageModel
from freshprice_env.freshprice_env import FreshPriceEnv
from freshprice_env.enums import CurriculumScenario

# Find latest checkpoint
checkpoints = sorted(glob.glob("checkpoints/grpo_level0/episode_*")) + \
              sorted(glob.glob("checkpoints/grpo_level0/final")) + \
              sorted(glob.glob("checkpoints/promoted_level*"))
latest = checkpoints[-1] if checkpoints else "checkpoints/sft_v1"
print(f"Evaluating: {latest}")

model_trained, tok_trained = FastLanguageModel.from_pretrained(
    model_name=latest, max_seq_length=4096, dtype=None, load_in_4bit=True)
FastLanguageModel.for_inference(model_trained)

def gen_trained(prompt):
    inputs = tok_trained(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model_trained.device)
    with torch.no_grad():
        out = model_trained.generate(**inputs, max_new_tokens=800, do_sample=False, pad_token_id=tok_trained.eos_token_id)
    return tok_trained.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

trained_results = []
for i in range(3):
    env = FreshPriceEnv(scenario=CurriculumScenario.STABLE_WEEK, seed=1000+i)
    obs, info = env.reset(seed=1000+i)
    done = False
    while not done:
        obs, reward, done, truncated, info = env.step(gen_trained(obs))
    final = info.get("final_reward", {})
    trained_results.append({"wrr": final.get("wrr", 0.0), "quality": final.get("brief_quality_score", 0.0)})
    print(f"Episode {i+1}: WRR={final.get('wrr',0):.4f} | Quality={final.get('brief_quality_score',0):.3f}")

print(f"\n{'='*50}")
print(f"FINAL RESULTS")
print(f"{'='*50}")
print(f"{'':>20} {'WRR':>8} {'Quality':>10}")
print(f"{'-'*40}")
print(f"{'Zero-shot':>20} {sum(r['wrr'] for r in baseline_results)/3:>8.4f} {sum(r['quality'] for r in baseline_results)/3:>10.3f}")
print(f"{'After SFT':>20} {sum(r['wrr'] for r in sft_results)/3:>8.4f} {sum(r['quality'] for r in sft_results)/3:>10.3f}")
print(f"{'After GRPO+DPO':>20} {sum(r['wrr'] for r in trained_results)/3:>8.4f} {sum(r['quality'] for r in trained_results)/3:>10.3f}")

## Cell 11: Save Results & Push to GitHub

In [ ]:
import json
import os

# Save results
os.makedirs("eval/plots", exist_ok=True)
results = {
    "baseline": baseline_results,
    "sft": sft_results,
    "trained": trained_results,
}
with open("eval/training_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Results saved to eval/training_results.json")

# Push to GitHub
!git add eval/training_results.json training/sft_data/
!git commit -m "results: baseline + SFT + GRPO training results"
!git push origin main
print("\nPushed to GitHub!")